# AI-Based Gardening Video Recommendation System

## Project Overview

This project creates a content-based recommendation system that recommends
gardening video topics based on a user's gardening interests.

The system considers the plant or garden area, gardening topic, experience
level, and an optional user question. It uses TF-IDF and cosine similarity
to identify gardening video topics that most closely match the user's needs.

The project adapts recommendation-system concepts from the movie
recommendation examples provided in the course and applies them to
home gardening.

### Technologies Used
- Python
- Pandas
- Scikit-learn
- TF-IDF
- Cosine Similarity
- Gradio

AI-Based Adaptive Gardening Recommendation Interface

In [1]:
# Import libraries used for data processing and recommendations

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load the gardening video dataset

videos = pd.read_csv("/content/gardening_videos.csv")

print("Dataset loaded successfully!")
print("Number of gardening topics:", len(videos))

Dataset loaded successfully!
Number of gardening topics: 74


In [3]:
# Display the first 10 gardening topics

videos.head(10)

,title,plant,category,topic,level,keywords
0,Growing Cucumbers Vertically on a Cattle Panel...,Cucumber,Vining Vegetables,Trellising,Beginner,cucumber vertical gardening cattle panel trell...
1,Pruning Cucumber Vines for Better Airflow,Cucumber,Vining Vegetables,Pruning,Intermediate,cucumber pruning vines airflow disease prevent...
2,Cucumber Watering Guide for Raised Beds,Cucumber,Vining Vegetables,Watering,Beginner,cucumber watering raised bed moisture mulch su...
3,Common Cucumber Pests and Organic Control,Cucumber,Vining Vegetables,Pest Control,Beginner,cucumber beetle aphids pests organic control neem
4,Why Cucumber Leaves Turn Yellow,Cucumber,Vining Vegetables,Disease,Beginner,cucumber yellow leaves nutrient deficiency ove...
5,Training Pole Beans on a Trellis,String Bean,Vining Vegetables,Trellising,Beginner,pole beans string beans trellis climbing vines...
6,Growing String Beans in Raised Beds,String Bean,Vining Vegetables,Planting,Beginner,string beans pole beans raised bed planting sp...
7,Harvesting String Beans for Continued Production,String Bean,Vining Vegetables,Harvesting,Beginner,string beans harvest picking production tender...
8,Squash Trellising and Vine Support,Squash,Vining Vegetables,Trellising,Intermediate,squash trellis vertical support vines fruit sl...
9,Squash Vine Borer Prevention and Control,Squash,Vining Vegetables,Pest Control,Intermediate,squash vine borer pest prevention organic control


In [4]:
# Check the dataset structure

videos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     74 non-null     object
 1   plant     74 non-null     object
 2   category  74 non-null     object
 3   topic     74 non-null     object
 4   level     74 non-null     object
 5   keywords  74 non-null     object
dtypes: object(6)
memory usage: 3.6+ KB


In [5]:
# Check for missing values

videos.isnull().sum()

,0
title,0
plant,0
category,0
topic,0
level,0
keywords,0


In [6]:
# Combine descriptive fields into one text feature

videos["combined_text"] = (
    videos["title"] + " " +
    videos["plant"] + " " +
    videos["category"] + " " +
    videos["topic"] + " " +
    videos["level"] + " " +
    videos["keywords"]
).str.lower()

videos[["title", "combined_text"]].head()

,title,combined_text
0,Growing Cucumbers Vertically on a Cattle Panel...,growing cucumbers vertically on a cattle panel...
1,Pruning Cucumber Vines for Better Airflow,pruning cucumber vines for better airflow cucu...
2,Cucumber Watering Guide for Raised Beds,cucumber watering guide for raised beds cucumb...
3,Common Cucumber Pests and Organic Control,common cucumber pests and organic control cucu...
4,Why Cucumber Leaves Turn Yellow,why cucumber leaves turn yellow cucumber vinin...


In [7]:
# Convert gardening text into numerical TF-IDF features

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(
    videos["combined_text"]
)

print("TF-IDF matrix created successfully!")
print("Matrix shape:", tfidf_matrix.shape)

TF-IDF matrix created successfully!
Matrix shape: (74, 963)


In [8]:
# Calculate similarity between all gardening topics

similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print("Similarity matrix created successfully!")
print("Matrix shape:", similarity_matrix.shape)

Similarity matrix created successfully!
Matrix shape: (74, 74)


In [9]:
# Create the gardening recommendation function

def recommend_gardening_videos(plant, topic, level, user_interest, number=5):

    # Combine the user's selections into one search query
    query_parts = []

    if plant and plant != "Any":
        query_parts.append(plant)

    if topic and topic != "Any":
        query_parts.append(topic)

    if level and level != "Any":
        query_parts.append(level)

    if user_interest:
        query_parts.append(user_interest)

    # Check if the user entered any information
    if len(query_parts) == 0:
        return "Please select or enter at least one gardening preference."

    # Combine the user's choices
    query_text = " ".join(query_parts).lower()

    # Convert the user's request into TF-IDF features
    query_vector = tfidf.transform([query_text])

    # Compare the request with all gardening topics
    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Rank the gardening topics from most similar to least similar
    ranked_indices = scores.argsort()[::-1]

    # Keep only results with a similarity score greater than zero
    recommendations = [
        index for index in ranked_indices
        if scores[index] > 0
    ][:int(number)]

    # Create the recommendation results
    results = []

    for rank, index in enumerate(recommendations, start=1):

        row = videos.iloc[index]

        results.append({
            "Rank": rank,
            "Recommended Video Topic": row["title"],
            "Plant": row["plant"],
            "Topic": row["topic"],
            "Level": row["level"],
            "Similarity": round(scores[index] * 100, 1)
        })

    return pd.DataFrame(results)

In [10]:
# Test 1: Cucumber trellising recommendation

recommend_gardening_videos(
    plant="Cucumber",
    topic="Trellising",
    level="Beginner",
    user_interest="cattle panel vertical gardening",
    number=5
)

,Rank,Recommended Video Topic,Plant,Topic,Level,Similarity
0,1,Growing Cucumbers Vertically on a Cattle Panel...,Cucumber,Trellising,Beginner,55.9
1,2,Using a Cattle Panel as a Garden Arch Trellis,General,Trellising,Beginner,42.1
2,3,Squash Trellising and Vine Support,Squash,Trellising,Intermediate,24.8
3,4,Cucumber Watering Guide for Raised Beds,Cucumber,Watering,Beginner,12.0
4,5,Why Cucumber Leaves Turn Yellow,Cucumber,Disease,Beginner,11.9


In [11]:
# Test 2: Tomato pest-control recommendation

recommend_gardening_videos(
    plant="Tomato",
    topic="Pest Control",
    level="Beginner",
    user_interest="hornworms and aphids",
    number=5
)

,Rank,Recommended Video Topic,Plant,Topic,Level,Similarity
0,1,Managing Aphids on Flowering Plants,Flowering Plants,Pest Control,Beginner,50.8
1,2,Tomato Hornworm and Aphid Control,Tomato,Pest Control,Beginner,47.6
2,3,Common Cucumber Pests and Organic Control,Cucumber,Pest Control,Beginner,34.2
3,4,Organic Pest Management for a Home Garden,General,Pest Control,Beginner,27.5
4,5,Squash Vine Borer Prevention and Control,Squash,Pest Control,Intermediate,23.8


In [12]:
# Test 3: Container citrus recommendation

recommend_gardening_videos(
    plant="Citrus",
    topic="Fertilizing",
    level="Intermediate",
    user_interest="potted lemon and orange trees",
    number=5
)

,Rank,Recommended Video Topic,Plant,Topic,Level,Similarity
0,1,Citrus Fertilizing Schedule for Container Trees,Citrus,Fertilizing,Intermediate,57.9
1,2,Growing Orange Trees in Pots,Orange,Container Growing,Beginner,34.8
2,3,Growing Lemon Trees in Containers,Lemon,Container Growing,Beginner,27.1
3,4,Pruning Container Citrus Trees,Citrus,Pruning,Beginner,18.1
4,5,Fertilizing Pepper Plants for More Fruit,Bell Pepper,Fertilizing,Intermediate,16.3


In [13]:
# Test 4: Blank input validation

recommend_gardening_videos(
    plant="Any",
    topic="Any",
    level="Any",
    user_interest="",
    number=5
)

'Please select or enter at least one gardening preference.'

## Improving the Recommendation Algorithm

Initial testing showed a limitation in the first version of the recommendation
system. For a search involving Tomato, Pest Control, Beginner, and
"hornworms and aphids," a general flowering-plant aphid topic ranked higher
than the more relevant tomato-specific result.

This occurred because TF-IDF and cosine similarity primarily measure textual
similarity and did not give additional importance to the user's selected
plant, topic, or experience level.

To improve the recommendations, the ranking function was modified to add
extra weight for exact matches:

- Plant match: +0.20
- Topic match: +0.15
- Experience-level match: +0.05

The updated system was then retested using the same input to determine
whether the modification improved the ranking.

In [14]:
# Improved gardening recommendation function
# Adds extra weight for exact plant, topic, and experience-level matches

def recommend_gardening_videos(plant, topic, level, user_interest, number=5):

    query_parts = []

    if plant and plant != "Any":
        query_parts.append(plant)

    if topic and topic != "Any":
        query_parts.append(topic)

    if level and level != "Any":
        query_parts.append(level)

    if user_interest:
        query_parts.append(user_interest)

    # Validate the user's input
    if len(query_parts) == 0:
        return "Please select or enter at least one gardening preference."

    # Combine user selections into one search query
    query_text = " ".join(query_parts).lower()

    # Convert the user's request into TF-IDF features
    query_vector = tfidf.transform([query_text])

    # Calculate text similarity
    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Add extra weight for exact category matches
    for index, row in videos.iterrows():

        if plant != "Any" and row["plant"] == plant:
            scores[index] += 0.20

        if topic != "Any" and row["topic"] == topic:
            scores[index] += 0.15

        if level != "Any" and row["level"] == level:
            scores[index] += 0.05

    # Rank recommendations
    ranked_indices = scores.argsort()[::-1]

    recommendations = [
        index for index in ranked_indices
        if scores[index] > 0
    ][:int(number)]

    # Prepare results
    results = []

    for rank, index in enumerate(recommendations, start=1):

        row = videos.iloc[index]

        results.append({
            "Rank": rank,
            "Recommended Video Topic": row["title"],
            "Plant": row["plant"],
            "Topic": row["topic"],
            "Level": row["level"],
            "Match Score": round(scores[index] * 100, 1)
        })

    return pd.DataFrame(results)

In [15]:
# Re-test the same tomato pest-control request after improving the algorithm

recommend_gardening_videos(
    plant="Tomato",
    topic="Pest Control",
    level="Beginner",
    user_interest="hornworms and aphids",
    number=5
)

,Rank,Recommended Video Topic,Plant,Topic,Level,Match Score
0,1,Tomato Hornworm and Aphid Control,Tomato,Pest Control,Beginner,87.6
1,2,Managing Aphids on Flowering Plants,Flowering Plants,Pest Control,Beginner,70.8
2,3,Common Cucumber Pests and Organic Control,Cucumber,Pest Control,Beginner,54.2
3,4,Organic Pest Management for a Home Garden,General,Pest Control,Beginner,47.5
4,5,How to Prune Tomato Suckers,Tomato,Pruning,Beginner,45.6


## Graphical User Interface

After testing and improving the recommendation algorithm, a graphical user
interface was created using Gradio. The interface allows users to interact
with the recommendation system without writing Python code.

Users can select a plant or garden area, choose the type of gardening help
needed, select an experience level, enter an optional gardening question,
and choose the number of recommendations they want to receive.

In [16]:
# Import Gradio for the graphical user interface

import gradio as gr

In [17]:
# Create dropdown options from the gardening dataset

plant_options = ["Any"] + sorted(videos["plant"].unique().tolist())

topic_options = ["Any"] + sorted(videos["topic"].unique().tolist())

level_options = ["Any", "Beginner", "Intermediate", "Advanced"]

print("Plant options:", len(plant_options))
print("Topic options:", len(topic_options))
print("Interface options created successfully!")

Plant options: 30
Topic options: 13
Interface options created successfully!


In [18]:
# Create the Gradio interface for the recommendation system

with gr.Blocks(title="AI Gardening Video Recommendation System") as demo:

    gr.Markdown("""
    # 🌱 AI-Based Gardening Video Recommendation System

    Select what you are growing and what you need help with.
    The system will recommend gardening video topics based on your interests.
    """)

    with gr.Row():

        plant_input = gr.Dropdown(
            choices=plant_options,
            value="Any",
            label="Plant or Garden Area"
        )

        topic_input = gr.Dropdown(
            choices=topic_options,
            value="Any",
            label="What Do You Need Help With?"
        )

        level_input = gr.Dropdown(
            choices=level_options,
            value="Any",
            label="Experience Level"
        )

    interest_input = gr.Textbox(
        label="Describe Your Gardening Question (Optional)",
        placeholder="Example: hornworms and aphids on my tomato plants"
    )

    number_input = gr.Slider(
        minimum=3,
        maximum=10,
        value=5,
        step=1,
        label="Number of Recommendations"
    )

    recommend_button = gr.Button(
        "Recommend Gardening Videos",
        variant="primary"
    )

    recommendation_output = gr.Dataframe(
        label="Recommended Gardening Video Topics"
    )

    recommend_button.click(
        fn=recommend_gardening_videos,
        inputs=[
            plant_input,
            topic_input,
            level_input,
            interest_input,
            number_input
        ],
        outputs=recommendation_output
    )

In [19]:
# Launch the final recommendation system

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://553c7abce6f6eb8f6f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
# Store user interaction history during the current session

user_profile = {
    "plants": {},
    "topics": {},
    "levels": {},
    "total_interactions": 0
}

In [21]:
# Update the user's session profile based on each interaction

def update_user_profile(plant, topic, level):

    user_profile["total_interactions"] += 1

    if plant and plant != "Any":
        user_profile["plants"][plant] = (
            user_profile["plants"].get(plant, 0) + 1
        )

    if topic and topic != "Any":
        user_profile["topics"][topic] = (
            user_profile["topics"].get(topic, 0) + 1
        )

    if level and level != "Any":
        user_profile["levels"][level] = (
            user_profile["levels"].get(level, 0) + 1
        )

    return user_profile

In [22]:
# Identify the user's most frequently selected preference

def get_learned_preference():

    if user_profile["total_interactions"] == 0:
        return "No preferences learned yet."

    preferred_plant = None
    preferred_topic = None

    if user_profile["plants"]:
        preferred_plant = max(
            user_profile["plants"],
            key=user_profile["plants"].get
        )

    if user_profile["topics"]:
        preferred_topic = max(
            user_profile["topics"],
            key=user_profile["topics"].get
        )

    message = "Learned preferences: "

    if preferred_plant:
        message += f"Plant = {preferred_plant}. "

    if preferred_topic:
        message += f"Topic = {preferred_topic}."

    return message

In [23]:
# Adapt the amount of guidance according to gardening experience level

def get_experience_guidance(level):

    if level == "Beginner":
        return (
            "🌱 Beginner Mode: Recommendations use simpler guidance "
            "and focus on basic gardening steps."
        )

    elif level == "Intermediate":
        return (
            "🌿 Intermediate Mode: Recommendations include additional "
            "care and maintenance information."
        )

    elif level == "Advanced":
        return (
            "🌳 Advanced Mode: Recommendations emphasize more detailed "
            "and specialized gardening topics."
        )

    else:
        return (
            "Select an experience level to personalize the interface."
        )

In [24]:
# Adapt result explanation according to experience level

def get_detail_message(level):

    if level == "Beginner":
        return (
            "Tip: Start with the highest-ranked recommendation. "
            "Focus on one gardening task at a time."
        )

    elif level == "Intermediate":
        return (
            "Tip: Compare several recommendations and consider "
            "watering, soil, pruning, and seasonal conditions."
        )

    elif level == "Advanced":
        return (
            "Advanced Tip: Compare recommendation scores and consider "
            "cultivar differences, soil conditions, pest pressure, "
            "fertilization, and local growing conditions."
        )

    return ""

In [25]:
# Adaptive recommendation function
# Combines the existing recommender with runtime personalization

def adaptive_gardening_recommendation(
    plant,
    topic,
    level,
    user_interest,
    number
):

    # Update interaction history
    update_user_profile(
        plant,
        topic,
        level
    )

    # Generate recommendations using the existing system
    recommendations = recommend_gardening_videos(
        plant,
        topic,
        level,
        user_interest,
        number
    )

    # Create adaptive messages
    experience_message = get_experience_guidance(level)
    preference_message = get_learned_preference()
    detail_message = get_detail_message(level)

    return (
        recommendations,
        experience_message,
        preference_message,
        detail_message
    )

In [26]:
adaptive_gardening_recommendation(
    plant="Tomato",
    topic="Pest Control",
    level="Beginner",
    user_interest="hornworms and aphids",
    number=5
)

(   Rank                    Recommended Video Topic             Plant  \
 0     1          Tomato Hornworm and Aphid Control            Tomato   
 1     2        Managing Aphids on Flowering Plants  Flowering Plants   
 2     3  Common Cucumber Pests and Organic Control          Cucumber   
 3     4  Organic Pest Management for a Home Garden           General   
 4     5                How to Prune Tomato Suckers            Tomato   
 
           Topic     Level  Match Score  
 0  Pest Control  Beginner         87.6  
 1  Pest Control  Beginner         70.8  
 2  Pest Control  Beginner         54.2  
 3  Pest Control  Beginner         47.5  
 4       Pruning  Beginner         45.6  ,
 '🌱 Beginner Mode: Recommendations use simpler guidance and focus on basic gardening steps.',
 'Learned preferences: Plant = Tomato. Topic = Pest Control.',
 'Tip: Start with the highest-ranked recommendation. Focus on one gardening task at a time.')

In [27]:
adaptive_gardening_recommendation(
    plant="Tomato",
    topic="Pruning",
    level="Beginner",
    user_interest="tomato suckers",
    number=5
)

(   Rank                   Recommended Video Topic   Plant         Topic  \
 0     1               How to Prune Tomato Suckers  Tomato       Pruning   
 1     2         Tomato Hornworm and Aphid Control  Tomato  Pest Control   
 2     3           Growing Tomatoes in Raised Beds  Tomato      Planting   
 3     4  Tomato Fertilizing from Flowers to Fruit  Tomato   Fertilizing   
 4     5    Preventing Blossom End Rot in Tomatoes  Tomato       Disease   
 
           Level  Match Score  
 0      Beginner        112.4  
 1      Beginner         51.7  
 2      Beginner         51.3  
 3  Intermediate         46.8  
 4      Beginner         44.0  ,
 '🌱 Beginner Mode: Recommendations use simpler guidance and focus on basic gardening steps.',
 'Learned preferences: Plant = Tomato. Topic = Pest Control.',
 'Tip: Start with the highest-ranked recommendation. Focus on one gardening task at a time.')

In [28]:
adaptive_gardening_recommendation(
    plant="Tomato",
    topic="Watering",
    level="Beginner",
    user_interest="watering tomato plants",
    number=5
)

(   Rank                   Recommended Video Topic     Plant         Topic  \
 0     1               How to Prune Tomato Suckers    Tomato       Pruning   
 1     2   Cucumber Watering Guide for Raised Beds  Cucumber      Watering   
 2     3         Tomato Hornworm and Aphid Control    Tomato  Pest Control   
 3     4    Preventing Blossom End Rot in Tomatoes    Tomato       Disease   
 4     5  Tomato Fertilizing from Flowers to Fruit    Tomato   Fertilizing   
 
           Level  Match Score  
 0      Beginner         55.3  
 1      Beginner         52.0  
 2      Beginner         48.9  
 3      Beginner         48.6  
 4  Intermediate         48.4  ,
 '🌱 Beginner Mode: Recommendations use simpler guidance and focus on basic gardening steps.',
 'Learned preferences: Plant = Tomato. Topic = Pest Control.',
 'Tip: Start with the highest-ranked recommendation. Focus on one gardening task at a time.')

In [29]:
get_learned_preference()

'Learned preferences: Plant = Tomato. Topic = Pest Control.'

In [30]:
get_experience_guidance("Advanced")

'🌳 Advanced Mode: Recommendations emphasize more detailed and specialized gardening topics.'

In [31]:
get_detail_message("Advanced")

'Advanced Tip: Compare recommendation scores and consider cultivar differences, soil conditions, pest pressure, fertilization, and local growing conditions.'

In [32]:
# Reset learned preferences for a new user session

def reset_user_profile():

    user_profile["plants"].clear()
    user_profile["topics"].clear()
    user_profile["levels"].clear()
    user_profile["total_interactions"] = 0

    return "User profile reset. No preferences learned yet."

In [33]:
# Build the Adaptive Gardening Recommendation Interface

import gradio as gr

# Create dropdown options from the gardening dataset
plant_options = ["Any"] + sorted(videos["plant"].dropna().unique().tolist())
topic_options = ["Any"] + sorted(videos["topic"].dropna().unique().tolist())
level_options = ["Any", "Beginner", "Intermediate", "Advanced"]


with gr.Blocks(title="Adaptive Gardening Recommendation Interface") as adaptive_demo:

    gr.Markdown(
        """
        # 🌱 Adaptive Gardening Recommendation Interface

        This system recommends gardening topics and adapts its guidance
        based on your experience level and interactions during the session.
        """
    )

    with gr.Row():

        plant_input = gr.Dropdown(
            choices=plant_options,
            value="Any",
            label="Plant or Garden Area"
        )

        topic_input = gr.Dropdown(
            choices=topic_options,
            value="Any",
            label="What Do You Need Help With?"
        )

        level_input = gr.Dropdown(
            choices=level_options,
            value="Any",
            label="Experience Level"
        )

    interest_input = gr.Textbox(
        label="Describe Your Gardening Question",
        placeholder="Example: hornworms and aphids on tomato plants"
    )

    number_input = gr.Slider(
        minimum=3,
        maximum=10,
        value=5,
        step=1,
        label="Number of Recommendations"
    )

    with gr.Row():

        recommend_button = gr.Button(
            "Get Adaptive Recommendations",
            variant="primary"
        )

        reset_button = gr.Button(
            "Reset Learned Preferences"
        )

    gr.Markdown("## Recommended Gardening Topics")

    recommendation_output = gr.Dataframe(
        label="Personalized Recommendations"
    )

    adaptive_status = gr.Markdown(
        "### Adaptive Interface Status\n"
        "Select your experience level to personalize the interface."
    )

    learned_preference_output = gr.Markdown(
        "### Learned Preferences\n"
        "No preferences learned yet."
    )

    adaptive_tip_output = gr.Markdown(
        "### Personalized Guidance\n"
        "Guidance will appear after your first recommendation."
    )


    # Generate recommendations and update adaptive information
    recommend_button.click(
        fn=adaptive_gardening_recommendation,
        inputs=[
            plant_input,
            topic_input,
            level_input,
            interest_input,
            number_input
        ],
        outputs=[
            recommendation_output,
            adaptive_status,
            learned_preference_output,
            adaptive_tip_output
        ]
    )


    # Reset learned preferences
    reset_button.click(
        fn=reset_user_profile,
        inputs=[],
        outputs=learned_preference_output
    )

In [34]:
adaptive_demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a7109b3387146dec29.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
